# MetaSort All-Tissue Deconvolution and Method Comparison

This notebook runs MetaSort on every tissue under `data/`, saves the deconvolved proportions, and compares MetaSort against existing Tabula-Sapiens result files for MuSiC and LinDeconSeq.

Metric per mixture:

`L1 = sum(abs(predicted_proportion - true_proportion))`

`Accuracy = 1 - L1 / 2`

The comparison uses `bulkRatio.txt` as ground truth and aligns cell types by name.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent]
REPO_ROOT = next((path for path in candidates if (path / "metasort").exists() and (path / "data").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the MetaSort repository root or notebooks directory.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from metasort import MetaSortConfig, MetaSortSolver

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Repo root: {REPO_ROOT}")

In [ ]:
DATA_ROOT = REPO_ROOT / "data"
TABULA_ROOT = Path("/home/yunhao/nfs/yunhao/nar_dataset_sim/Tabula-Sapiens")

tissue_names = sorted([path.name for path in DATA_ROOT.iterdir() if path.is_dir()])
n_mixtures = 100
RECOMPUTE_METASORT = False

metasort_deconv_out = REPO_ROOT / "metasort_all_tissues_deconvolution.csv"
metasort_deconv_long_out = REPO_ROOT / "metasort_all_tissues_deconvolution_long.csv"
metasort_metrics_out = REPO_ROOT / "metasort_all_tissues_metrics.csv"
comparison_metrics_out = REPO_ROOT / "method_comparison_all_tissues_metrics.csv"
comparison_summary_out = REPO_ROOT / "method_comparison_all_tissues_summary.csv"

method_files = {
    "MuSiC": "MuSic.txt",
    "LinDeconSeq": "LinDeconSeq_from_notebook.txt",
}

cfg = MetaSortConfig(
    lambda_hessian=1.0,
    lambda_avg_gradient=1.0,
    lambda_residual=1.5,
    lambda_gene_importance=0.0,
    lambda3=0.01,
    lambda4=0.001,
    convergence_tol=0.005,
    print_info=False,
)
solver = MetaSortSolver(cfg)

tissue_names

In [ ]:
def normalize_proportions(values: np.ndarray) -> np.ndarray:
    values = np.clip(np.asarray(values, dtype=float), 0.0, None)
    total = float(values.sum())
    if total <= 0:
        return np.full(values.shape, 1.0 / len(values), dtype=float)
    return values / total


def l1_and_accuracy(pred: np.ndarray, truth: np.ndarray) -> tuple[float, float]:
    pred = normalize_proportions(pred)
    truth = normalize_proportions(truth)
    l1_error = float(np.sum(np.abs(pred - truth)))
    accuracy = float(1.0 - l1_error / 2.0)
    return l1_error, accuracy


def load_tissue_inputs(tissue_name: str):
    tissue_root = DATA_ROOT / tissue_name
    signature_df = pd.read_csv(tissue_root / "signature.txt", sep="\t", index_col=0)
    bulk_df = pd.read_csv(tissue_root / "bulk.txt", sep="\t", index_col=0)
    ratio_df = pd.read_csv(tissue_root / "bulkRatio.txt", sep="\t", index_col=0)

    common_genes = signature_df.index.intersection(bulk_df.index)
    signature_df = signature_df.loc[common_genes]
    bulk_df = bulk_df.loc[common_genes]
    cell_types = signature_df.columns.to_list()
    mixture_names = list(bulk_df.columns[:n_mixtures])
    return signature_df, bulk_df, ratio_df, cell_types, mixture_names


def run_metasort_tissue(tissue_name: str) -> tuple[list[dict], list[dict]]:
    signature_df, bulk_df, ratio_df, cell_types, mixture_names = load_tissue_inputs(tissue_name)
    signature = signature_df.to_numpy(dtype=float)
    deconv_rows = []
    metric_rows = []

    for mixture_name in mixture_names:
        bulk = bulk_df[mixture_name].to_numpy(dtype=float)
        truth = ratio_df.loc[cell_types, mixture_name].to_numpy(dtype=float)
        truth = normalize_proportions(truth)
        result = solver.solve(signature, bulk, cell_types=cell_types)
        pred = normalize_proportions(np.asarray(result.proportions, dtype=float))
        l1_error, accuracy = l1_and_accuracy(pred, truth)

        row = {
            "Tissue": tissue_name,
            "Mixture": mixture_name,
            "Method": "MetaSort",
            "L1_Error": l1_error,
            "Accuracy": accuracy,
            "Iterations": int(result.iterations),
            "Converged": bool(result.converged),
        }
        for cell_type, value in zip(cell_types, pred):
            row[f"Pred_{cell_type}"] = float(value)
        for cell_type, value in zip(cell_types, truth):
            row[f"Truth_{cell_type}"] = float(value)
        deconv_rows.append(row)

        metric_rows.append({
            "Tissue": tissue_name,
            "Mixture": mixture_name,
            "Method": "MetaSort",
            "L1_Error": l1_error,
            "Accuracy": accuracy,
        })

    return deconv_rows, metric_rows

In [ ]:
if metasort_deconv_out.exists() and metasort_metrics_out.exists() and not RECOMPUTE_METASORT:
    metasort_deconv_df = pd.read_csv(metasort_deconv_out)
    metasort_metrics_df = pd.read_csv(metasort_metrics_out)
else:
    deconv_rows = []
    metric_rows = []
    for tissue_name in tissue_names:
        tissue_deconv_rows, tissue_metric_rows = run_metasort_tissue(tissue_name)
        deconv_rows.extend(tissue_deconv_rows)
        metric_rows.extend(tissue_metric_rows)

    metasort_deconv_df = pd.DataFrame(deconv_rows)
    metasort_metrics_df = pd.DataFrame(metric_rows)
    metasort_deconv_df.to_csv(metasort_deconv_out, index=False)
    metasort_metrics_df.to_csv(metasort_metrics_out, index=False)

print(metasort_deconv_out)
metasort_deconv_df.head()

In [ ]:
long_rows = []
for _, row in metasort_deconv_df.iterrows():
    pred_columns = [col for col in metasort_deconv_df.columns if col.startswith("Pred_") and pd.notna(row[col])]
    for pred_col in pred_columns:
        cell_type = pred_col.removeprefix("Pred_")
        truth_col = f"Truth_{cell_type}"
        if truth_col not in metasort_deconv_df.columns or pd.isna(row[truth_col]):
            continue
        long_rows.append({
            "Tissue": row["Tissue"],
            "Mixture": row["Mixture"],
            "CellType": cell_type,
            "PredictedProportion": float(row[pred_col]),
            "TrueProportion": float(row[truth_col]),
        })

metasort_deconv_long_df = pd.DataFrame(long_rows)
metasort_deconv_long_df.to_csv(metasort_deconv_long_out, index=False)
print(metasort_deconv_long_out)
metasort_deconv_long_df.head()

In [ ]:
def normalize_mixture_name(name: str) -> str:
    name = str(name).strip().strip('"')
    if name.startswith("Bulk_"):
        return "Mixture" + name.split("_", 1)[1]
    return name


def read_external_method_result(tissue_name: str, method_name: str) -> pd.DataFrame:
    result_path = TABULA_ROOT / tissue_name / "result" / method_files[method_name]
    result_df = pd.read_csv(result_path, sep="\t", index_col=0)
    result_df.index = [normalize_mixture_name(idx) for idx in result_df.index]
    result_df.columns = [str(col).strip().strip('"') for col in result_df.columns]
    return result_df


def evaluate_external_method(tissue_name: str, method_name: str) -> list[dict]:
    _, _, ratio_df, cell_types, mixture_names = load_tissue_inputs(tissue_name)
    result_df = read_external_method_result(tissue_name, method_name)
    rows = []

    for mixture_name in mixture_names:
        if mixture_name not in result_df.index:
            continue
        available_cell_types = [cell_type for cell_type in cell_types if cell_type in result_df.columns and cell_type in ratio_df.index]
        pred = result_df.loc[mixture_name, available_cell_types].to_numpy(dtype=float)
        truth = ratio_df.loc[available_cell_types, mixture_name].to_numpy(dtype=float)
        l1_error, accuracy = l1_and_accuracy(pred, truth)
        rows.append({
            "Tissue": tissue_name,
            "Mixture": mixture_name,
            "Method": method_name,
            "L1_Error": l1_error,
            "Accuracy": accuracy,
        })
    return rows


if comparison_metrics_out.exists() and comparison_summary_out.exists() and not RECOMPUTE_METASORT:
    comparison_df = pd.read_csv(comparison_metrics_out)
    summary_df = pd.read_csv(comparison_summary_out)
else:
    external_rows = []
    for tissue_name in tissue_names:
        for method_name in method_files:
            external_rows.extend(evaluate_external_method(tissue_name, method_name))

    comparison_df = pd.concat([metasort_metrics_df, pd.DataFrame(external_rows)], ignore_index=True)
    summary_df = (
        comparison_df
        .groupby(["Tissue", "Method"], as_index=False)
        .agg(
            NMixtures=("Mixture", "nunique"),
            MeanL1=("L1_Error", "mean"),
            MedianL1=("L1_Error", "median"),
            MeanAccuracy=("Accuracy", "mean"),
            MedianAccuracy=("Accuracy", "median"),
        )
    )
    comparison_df.to_csv(comparison_metrics_out, index=False)
    summary_df.to_csv(comparison_summary_out, index=False)

summary_df.sort_values(["Tissue", "MeanAccuracy"], ascending=[True, False]).round(4)

In [ ]:
method_order = ["MetaSort", "MuSiC", "LinDeconSeq"]
plot_tissue_names = ["Fat", "Lung"]
plot_tissue_labels = [f"{tissue} (100 bulk)" for tissue in plot_tissue_names]
plot_summary = summary_df.copy()
plot_summary["Method"] = pd.Categorical(plot_summary["Method"], categories=method_order, ordered=True)
plot_summary = plot_summary.sort_values(["Tissue", "Method"])

fig, axes = plt.subplots(1, len(plot_tissue_names), figsize=(4.4 * len(plot_tissue_names), 4.2), sharey=True)
if len(plot_tissue_names) == 1:
    axes = [axes]

colors = {
    "MetaSort": "#2563eb",
    "MuSiC": "#16a34a",
    "LinDeconSeq": "#f97316",
}

for ax, tissue_name, tissue_label in zip(axes, plot_tissue_names, plot_tissue_labels):
    tissue_df = plot_summary.loc[plot_summary["Tissue"] == tissue_name].set_index("Method").reindex(method_order).reset_index()
    values = tissue_df["MeanL1"].to_numpy(dtype=float)
    bars = ax.bar(
        np.arange(len(method_order)),
        values,
        color=[colors[method] for method in method_order],
        edgecolor="#263238",
        linewidth=0.8,
    )
    for bar, value in zip(bars, values):
        if np.isnan(value):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.012,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            color="#263238",
        )
        ax.set_title(tissue_label)
    ax.set_xticks(np.arange(len(method_order)), method_order, rotation=35, ha="right")
    ax.set_ylim(0.0, max(0.55, np.nanmax(values) * 1.25))
    ax.grid(axis="y", alpha=0.25)

axes[0].set_ylabel("Mean L1 Error")
fig.suptitle("Mean L1 Error: MetaSort vs Existing Tabula-Sapiens Results", y=1.03, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plot_tissue_names = ["Fat", "Lung"]
plot_tissue_labels = [f"{tissue} (100 bulk)" for tissue in plot_tissue_names]
fig, ax = plt.subplots(figsize=(6.2, 4.6))
width = 0.18
x = np.arange(len(plot_tissue_names))

for offset_idx, method_name in enumerate(method_order):
    method_values = []
    for tissue_name in plot_tissue_names:
        match = summary_df.loc[(summary_df["Tissue"] == tissue_name) & (summary_df["Method"] == method_name), "MeanL1"]
        method_values.append(float(match.iloc[0]) if len(match) else np.nan)
    ax.bar(
        x + (offset_idx - 1.5) * width,
        method_values,
        width=width,
        label=method_name,
        color=colors[method_name],
        edgecolor="#263238",
        linewidth=0.7,
    )

ax.set_xticks(x, plot_tissue_labels)
ax.set_ylabel("Mean L1 Error")
ax.set_ylim(0.0, max(0.55, np.nanmax(values) * 1.25))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.14))
plt.tight_layout()
plt.show()

In [ ]:
comparison_metrics_out, comparison_summary_out, metasort_deconv_out, metasort_metrics_out